# Grouped control predicate

In [ ]:
from collections import Counter

from IPython.display import SVG, display

from examples.algorithms.control_predicate_catalog import predicate_examples
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler.registers.control.control_predicate_resource_model import (
    control_predicate_resource_formula,
)
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

## Control specification

Each clause compares one integer or bit-array group with a constant. The target
is one precisely when all comparisons succeed. Control registers are passed
through. Words use big-endian bit order and arrays use row-major order. The
predicate can drive multiple operations before its inverse releases the work.

- `ctrl_i` is the i-th control group.
- `junk` owns the first `n - 2` prefix outputs for `n` control bits. Binary
  predicates omit this port.
- `target` owns the final prefix output.

The measured inverse consumes junk and target and clears those wires. See
[Unqomp: synthesizing uncomputation in Quantum circuits](https://doi.org/10.1145/3453483.3454040).

In [ ]:
examples = predicate_examples()

### Example instances

In [ ]:
ctrl_on_int = examples["ctrl_on_int"]

In [ ]:
ctrl_on_bits = examples["ctrl_on_bits"]

In [ ]:
ctrl_on_nd_bits = examples["ctrl_on_nd_bits"]

In [ ]:
ctrl_on_multiple_values = examples["ctrl_on_multiple_values"]

In [ ]:
ctrl_on_symbolic_cv = examples["ctrl_on_symbolic_cv"]

In [ ]:
ctrl_on_symbolic_cv_multi = examples["ctrl_on_symbolic_cv_multi"]

### Graphical signatures

The final two diagrams have concrete group sizes with unspecified activation values.

In [ ]:
display(SVG(field_signature_svg([
    (name, example.signature, {})
    for name, example in examples.items()
    if name != "ctrl_on_symbolic_n_ctrls"
])))

### One-level call graph

In [ ]:
child = ctrl_on_int.source_child()
calls = {("Control predicate", f"MultiAnd{child.controls}"): 1}
display(SVG(render_call_graph_svg(export_call_graph_dot("Control predicate", calls))))
display({f"MultiAnd{child.controls}": 1})

### Execution and symbolic estimates

In [ ]:
circuit = DynamicCircuit(7).append(ctrl_on_int).append(ctrl_on_int.adjoint())
display(Counter(gate.name for gate in circuit.to_ir().instructions))

symbolic = examples["ctrl_on_symbolic_n_ctrls"]
print([(port.name, port.shape_symbolic, port.bitsize) for port in symbolic.signature])
display(control_predicate_resource_formula(symbolic.activation))